# 🎛 --enforce-sampling の全パターン検証（モック上流）

temp=0 / rp<1.05 / 欠落 / 正常値 を通して、上流に何が渡るかを見る。


In [ ]:
# --enforce-sampling の全パターン検証（モック上流・GPU 不要）
import base64
import json
import os
import socket
import subprocess
import sys
import threading
import time
import urllib.request
from http.server import BaseHTTPRequestHandler, HTTPServer

UP_PORT = 18000
PROXY_PORT = 18435

# 上流: 受け取った sampling パラメータをそのまま返す
SEEN = []


class Up(BaseHTTPRequestHandler):
    def do_POST(self):
        n = int(self.headers.get("content-length") or 0)
        body = json.loads(self.rfile.read(n) or b"{}")
        SEEN.append({k: body.get(k) for k in
                     ("temperature", "repetition_penalty", "top_p", "top_k")})
        out = json.dumps({"id": "x", "object": "chat.completion",
                          "choices": [{"index": 0, "message": {"role": "assistant",
                                     "content": "ok"}, "finish_reason": "stop"}],
                          "usage": {"prompt_tokens": 1, "completion_tokens": 1,
                                    "total_tokens": 2}}).encode()
        self.send_response(200)
        self.send_header("content-type", "application/json")
        self.send_header("content-length", str(len(out)))
        self.end_headers()
        self.wfile.write(out)

    def log_message(self, *a):
        pass


srv = HTTPServer(("127.0.0.1", UP_PORT), Up)
threading.Thread(target=srv.serve_forever, daemon=True).start()

os.makedirs("/tmp/px", exist_ok=True)
open("/tmp/px/auth_proxy.py", "wb").write(base64.b64decode("..."))

CASES = [
    {},
    {"temperature": 0},
    {"temperature": 0.0, "repetition_penalty": 1.0},
    {"temperature": 0.2, "repetition_penalty": 1.01},
    {"temperature": 1.5, "repetition_penalty": 1.05},
    {"temperature": 0.7, "repetition_penalty": 1.7},
    {"top_p": 0.9, "top_k": 20},
]


def run_proxy(extra):
    p = subprocess.Popen([sys.executable, "/tmp/px/auth_proxy.py",
                          "--upstream", f"http://127.0.0.1:{UP_PORT}",
                          "--port", str(PROXY_PORT)] + extra,
                         stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    for _ in range(50):
        time.sleep(0.2)
        try:
            socket.create_connection(("127.0.0.1", PROXY_PORT), 0.5).close()
            return p
        except OSError:
            continue
    return p


def post(payload):
    data = json.dumps({"model": "m", "messages": [{"role": "user",
                       "content": "hi"}], **payload}).encode()
    req = urllib.request.Request(f"http://127.0.0.1:{PROXY_PORT}/v1/chat/completions",
                                 data=data, headers={"content-type": "application/json"})
    with urllib.request.urlopen(req, timeout=20) as r:
        r.read()


for extra in ([], ["--enforce-sampling"]):
    SEEN.clear()
    p = run_proxy(extra)
    label = "enforce" if extra else "passthrough"
    print(f"\n===== {label} =====")
    for c in CASES:
        try:
            post(c)
        except Exception as e:
            print("REQ NG", c, type(e).__name__, e)
            continue
        print(f"  in={c} -> up={SEEN[-1]}")
    p.terminate()
    p.wait(timeout=10)
    time.sleep(0.5)

srv.shutdown()
print("\nDONE")
